# Round 11 — Within-class support subtypes

Two full independent families, frozen before execution. Cached vectors only; no model download, no new neural inference. This is exploratory development, not a Kaggle score.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "configs/multiprototype_features.json").is_file())
from scripts.run_multiprototype_features import figures, write_dashboard
SPEC = json.loads((ROOT / "configs/multiprototype_features.json").read_text())
RESULT = json.loads((ROOT / "reports/multiprototype_features/results.json").read_text())
print("Registered primary:", SPEC["primary"])
print("New classifier fits:", RESULT["new_fits"])
print("Prior readouts reused:", RESULT["reused_prior_controls"])
CHARTS = figures(RESULT)

Registered primary: prototype_all
New classifier fits: 12
Prior readouts reused: 10


## 1. Hypothesis and matched evidence

Do multiple bounded, reference-only subtypes within each supplied class help beyond a single mean and matched random class partitions?

The anchor was selected after Round 9 and is not promoted. All candidates retain the exact same saved anchor features and classifier.

In [2]:
display(pd.DataFrame(RESULT["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")

,fold,policy,variant,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_raw,0.679254,0.250238,0.760003
1,1,No legal advice: Do not offer or request legal...,qwen_raw,0.760533,0.221528,0.709879
2,0,"No Advertising: Spam, referral links, unsolici...",answer_only,0.679254,0.260747,0.828707
3,1,No legal advice: Do not offer or request legal...,answer_only,0.760533,0.228357,0.767258
4,0,"No Advertising: Spam, referral links, unsolici...",frozen_basic,0.693097,0.254469,0.827292
5,1,No legal advice: Do not offer or request legal...,frozen_basic,0.753038,0.234327,0.821515
6,0,"No Advertising: Spam, referral links, unsolici...",context_evidence,0.703470,0.247081,0.823188
7,1,No legal advice: Do not offer or request legal...,context_evidence,0.755044,0.235380,0.840367
8,0,"No Advertising: Spam, referral links, unsolici...",uniform_all,0.703246,0.247805,0.821082
9,1,No legal advice: Do not offer or request legal...,uniform_all,0.754574,0.233871,0.817539


## 2. Conditional uncertainty

A mean improvement can hide a policy regression. Paired intervals cover this round, not all adaptive research choices or future policies.

In [3]:
display(pd.DataFrame(RESULT["comparisons"]))
CHARTS[1].show(renderer="plotly_mimetype")

,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,prototype_location,context_evidence,prototype_location vs context anchor,-0.006612,-0.029526,0.016302
1,prototype_coverage,context_evidence,prototype_coverage vs context anchor,-0.013009,-0.035923,0.009905
2,prototype_all,context_evidence,prototype_all vs context anchor,-0.012963,-0.035877,0.009951
3,random_partition_all,context_evidence,random_partition_all vs context anchor,-0.013044,-0.035958,0.009870
4,single_center_all,context_evidence,single_center_all vs context anchor,-0.003913,-0.026827,0.019001
5,label_null_all,context_evidence,label_null_all vs context anchor,-0.032302,-0.055216,-0.009388
6,prototype_all,qwen_raw,Primary vs qwen_raw,-0.003599,-0.026513,0.019315
7,prototype_all,frozen_basic,Primary vs frozen_basic,-0.006774,-0.029688,0.016140
8,prototype_all,uniform_all,Primary vs uniform_all,-0.012616,-0.035530,0.010298
9,prototype_all,random_partition_all,Primary vs random_partition_all,0.000081,-0.022833,0.022995


## 3. Mechanism-specific controls

random_partition_all; single_center_all; label_null_all. Controls diagnose attribution; no secondary winner may silently replace the primary.

In [4]:
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

## 4. Reference coverage and failure modes

Exactly 36 new columns are proposed. Vocabulary or class-mode limitations are reported, not hidden. No query labels enter these descriptors.

In [5]:
display(pd.DataFrame(RESULT["diagnostics"]))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

,fold,inner_fold,self_overlap,mode,class,reference_rows,prototype_count,smallest_mode_fraction,largest_mode_fraction,mean_radius,outside_all_modes_fraction,query_used_for_clustering
0,0,0,0,clustered,0,250,4,0.020000,0.632000,0.05,0.000000,False
1,0,0,0,clustered,1,169,4,0.005917,0.869822,0.05,0.000000,False
2,0,0,0,random_partition,0,250,4,0.020000,0.632000,0.05,0.052381,False
3,0,0,0,random_partition,1,169,4,0.005917,0.869822,0.05,0.104762,False
4,0,0,0,single_center,0,250,1,1.000000,1.000000,0.05,0.052381,False
...,...,...,...,...,...,...,...,...,...,...,...,...
59,1,outer_query,0,random_partition,1,218,4,0.059633,0.458716,0.05,0.188563,False
60,1,outer_query,0,single_center,0,148,1,1.000000,1.000000,0.05,0.119011,False
61,1,outer_query,0,single_center,1,218,1,1.000000,1.000000,0.05,0.190108,False
62,1,outer_query,0,label_null,0,148,4,0.033784,0.574324,0.05,0.000000,False


## 5. Fitted associations, not causal importance

These coefficients summarize the trained linear readout. Matched ablations, not coefficient magnitude, determine evidence for a family.

In [6]:
CHARTS[6].show(renderer="plotly_mimetype")

## 6. Probability quality and metric definitions

Macro policy AUC, raw pooled AUC and within-policy-ranked pooled AUC are different diagnostics. None is a new hidden evaluation score.

In [7]:
display(pd.DataFrame(RESULT["pooled_metrics"]))
CHARTS[7].show(renderer="plotly_mimetype")

,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,context_evidence,0.729257,0.736183,0.741330
4,uniform_all,0.728910,0.735914,0.740913
5,prototype_location,0.722645,0.735492,0.737638
6,prototype_coverage,0.716248,0.735898,0.734094
7,prototype_all,0.716294,0.735982,0.734279
8,random_partition_all,0.716213,0.725430,0.725551
9,single_center_all,0.725344,0.736457,0.739801


## 7. Preregistered decision and remaining gaps

Primary must beat every specified comparator, without a policy regression. Adapted support-training answer margins remain in-sample: feature cross-fitting does not fix encoder-level dependence. The companion round is independent.

Research motivation: https://proceedings.mlr.press/v97/allen19b.html

In [8]:
print("Decision:", RESULT["decision"])
for item in RESULT["primary_requirements"]:
    print(item["reference"], item["per_policy_delta"], "passed:", item["passed"])
for limitation in RESULT["limitations"]:
    print(limitation)
print("Interactive dashboard:", write_dashboard(ROOT, RESULT))

Decision: DO_NOT_PROMOTE_PRIMARY
qwen_raw [-0.0014552238805970452, -0.005743527523923042] passed: False
frozen_basic [-0.015298507462686683, 0.001751433435744909] passed: False
context_evidence [-0.025671641791044753, -0.0002543981526779149] passed: False
uniform_all [-0.025447761194029805, 0.00021525997534310726] passed: False
random_partition_all [-0.018507462686567222, 0.01866891058883391] passed: False
single_center_all [-0.016641791044776033, -0.0014578971057316314] passed: False
label_null_all [0.006194029850746352, 0.03248468718811781] passed: False
Exploratory same-cohort follow-up; no independent holdout or Kaggle score.
Both new designs were frozen before either new result; they are independent.
The context_evidence anchor was selected after Round 9 and was not promoted.
Reference features are group cross-fitted, but the adapted answer margin is in-sample.
Cross-fitting new features does not remove the inherited answer-score limitation.
Intervals cover planned within-round co